In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer


In [3]:
df = pd.read_csv('../data/cleaned/cleaned_data.csv')

In [4]:
df['Profit_Flag'] = df['ROI'].apply(lambda x: 'Profit' if x > 0 else 'Loss')

In [5]:
df.head()

,Brand,Date,Campaign_Type,Target_Audience,Channel_Used,Language,Customer_Segment,Duration,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Engagement_Score,Total_Cost,Profit_Flag
0,Nykaa,2025-04-29,Social Media,College Students,"Whatsapp, Youtube",Hindi,College Students,21,57804,6156,3616,2355,1867515,209,2.79,20.98,492195,Profit
1,Nykaa,2025-04-06,Paid Ads,Tier 2 City Customers,Youtube,Hindi,College Students,18,91801,3321,1971,1357,1046247,180,3.28,7.24,244260,Profit
2,Nykaa,2025-01-14,Influencer,Youth,"Whatsapp, Google, Youtube",English,College Students,23,15536,2182,952,755,197055,90,1.90,25.03,67950,Profit
3,Nykaa,2025-06-04,Email,Working Women,"Youtube, Facebook, Instagram",Hindi,College Students,18,88114,8413,2231,947,376906,249,0.60,13.15,235803,Profit
4,Nykaa,2024-12-29,Paid Ads,College Students,"Facebook, Instagram",Hindi,Tier 2 City Customers,10,96871,3743,2060,1258,518296,228,0.81,7.29,286824,Profit


In [6]:
df['Profit_Flag'].value_counts()

Profit_Flag
Profit    128130
Loss       38157
Name: count, dtype: int64

In [7]:
df['Profit_Flag'].value_counts(normalize=True) * 100

Profit_Flag
Profit    77.053528
Loss      22.946472
Name: proportion, dtype: float64

In [8]:
channels_split = df['Channel_Used'].str.split(',').apply(lambda x: [c.strip() for c in x])

In [9]:
print(channels_split)

0                    [Whatsapp, Youtube]
1                              [Youtube]
2            [Whatsapp, Google, Youtube]
3         [Youtube, Facebook, Instagram]
4                  [Facebook, Instagram]
                       ...              
166282       [Instagram, Youtube, Email]
166283               [Google, Instagram]
166284             [Facebook, Instagram]
166285                   [Google, Email]
166286                       [Instagram]
Name: Channel_Used, Length: 166287, dtype: object


In [10]:
mlb = MultiLabelBinarizer()
channel_encoded = pd.DataFrame(
    np.asarray(mlb.fit_transform(channels_split)),
    columns=mlb.classes_,
    index=df.index
)

df = pd.concat([df, channel_encoded], axis=1)

In [11]:
df[mlb.classes_].sum()

Email        55470
Facebook     55414
Google       55491
Instagram    55479
Whatsapp     55358
Youtube      55480
dtype: int64

In [12]:
df[['Channel_Used'] + list(mlb.classes_)].head()  

,Channel_Used,Email,Facebook,Google,Instagram,Whatsapp,Youtube
0,"Whatsapp, Youtube",0,0,0,0,1,1
1,Youtube,0,0,0,0,0,1
2,"Whatsapp, Google, Youtube",0,0,1,0,1,1
3,"Youtube, Facebook, Instagram",0,1,0,1,0,1
4,"Facebook, Instagram",0,1,0,1,0,0


In [13]:
df.drop(columns=['Channel_Used'], inplace=True)

In [14]:
cat_cols = ['Brand', 'Campaign_Type', 'Target_Audience', 'Customer_Segment', 'Language']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 166287 entries, 0 to 166286
Data columns (total 35 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   Date                                    158209 non-null  str    
 1   Duration                                166287 non-null  int64  
 2   Impressions                             166287 non-null  int64  
 3   Clicks                                  166287 non-null  int64  
 4   Leads                                   166287 non-null  int64  
 5   Conversions                             166287 non-null  int64  
 6   Revenue                                 166287 non-null  int64  
 7   Acquisition_Cost                        166287 non-null  int64  
 8   ROI                                     166287 non-null  float64
 9   Engagement_Score                        166287 non-null  float64
 10  Total_Cost                              166287 non-null

In [16]:
df.to_csv('../data/processed/model_ready_data.csv', index=False)